In [1]:
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [2]:
from postprocess.ia.eval import (
    answers_for_model,
    fm_scores_for_model,
    metrics_for_model,
    metrics_for_models
)

In [3]:
MODEL = "qwen2_5_vl_7b"
task1 = ("strokerehab_ia1_3_30", "strokerehab_ia1_31_33")  # simultaneous
task2 = ("strokerehab_ia2_3_30", "strokerehab_ia2_31_33")  # individual

print(metrics_for_model("qwen2_5_vl_7b", tasks=task1))
print(metrics_for_model("qwen2_5_vl_7b", tasks=task2))

{'accuracy': 0.325, 'apd': 29.5, 'mtsd': 28.0}
{'accuracy': 0.041666666666666664, 'apd': 55.5, 'mtsd': 55.0}


In [4]:
df = fm_scores_for_model(MODEL, tasks=task1)

def pivot_fm_scores(df):
    df_long = df.melt(
        id_vars=['patient', 'fm_item'],
        value_vars=['pred_score', 'gt_score'],
        var_name='score_type',
        value_name='score'
    )
    df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
    out = df_long.pivot(index="fm_item", columns="col", values="score")
    return out

pivot_fm_scores(df)

col,gt_score_C00011,gt_score_S0001,gt_score_S00021,gt_score_S0005,pred_score_C00011,pred_score_S0001,pred_score_S00021,pred_score_S0005
fm_item,,,,,,,,
3,2,2,1,2,1,1,0,0
4,2,2,0,2,1,2,1,1
5,2,2,1,2,2,1,1,1
6,2,2,0,2,2,2,2,2
7,2,2,0,2,2,2,2,2
8,2,1,0,2,2,2,1,1
9,2,2,0,2,1,1,1,1
10,2,1,0,2,1,1,1,1
11,2,2,0,2,1,1,1,1


In [5]:
df = answers_for_model(MODEL, tasks=task1)

In [6]:
def pivot_answers(df):
    melted_df = df.melt(
        id_vars=['patient', 'qid'],
        value_vars=['answer'],
        var_name='answer_type',
    )
    melted_df.drop(columns=['answer_type'], inplace=True)
    melted_df.rename({'value': 'answer'}, axis=1, inplace=True)
    out = (
        melted_df
        .pivot(index="qid", columns="patient", values="answer")
        .add_suffix("_answer")
    )
    return out

# df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
# out = df_long.pivot(index="fm_item", columns="col", values="score")

In [48]:
pivot_answers(df).to_csv("scrap.csv", index=True)

In [33]:
df = fm_scores_for_model(MODEL, tasks=("strokerehab_ia3_3_30",))
pivot_fm_scores(df)

ValueError: Duplicate entry for (patient=C00011, qid=10).

In [34]:
df = answers_for_model(model=MODEL, tasks=("strokerehab_ia3_3_30",))
patient_order = ['C00011', 'S0005', 'S0001', 'S00021']
patient_order = [p + '_answer' for p in patient_order]
pivot_answers(df)[patient_order]

ValueError: Duplicate entry for (patient=C00011, qid=10).

In [32]:
import pandas as pd
from data.utils_strokerehab import DataPaths
pd.set_option("display.max_colwidth", None)
df = pd.read_csv("./data/ia/fm_item_questions3.csv")
df[df['fm_video'].str.contains("13")][['qid','question']]

,qid,question
0,10,Is the arm down at the side at onset? Answer 'Yes' or 'No' directly.
1,11,"At the beginning, is the arm down at the side? Answer 'Yes' or 'No' directly."
2,20,Is the elbow fully extended (straight) at onset? Answer 'Yes' or 'No' directly.
3,21,Is the elbow straight at onset? Answer 'Yes' or 'No' directly.
4,22,"At the beginning, is the elbow straight? Answer 'Yes' or 'No' directly."
5,23,"At the beginning, is the elbow mostly straight? Answer 'Yes' or 'No' directly."
6,24,Is the elbow mostly straight? Answer 'Yes' or 'No' directly.
7,30,Is the forearm in a neutral position (thumb pointing forward) at onset? Answer 'Yes' or 'No' directly.
8,31,"At the beginning, is the forearm in a neutral position (thumb pointing forward)? Answer 'Yes' or 'No' directly."
9,32,"At the beginning, is the forearm in a neutral position (thumb pointing in the same direction that the person is facing)? Answer 'Yes' or 'No' directly."
